 Tableau 2 : Analyse Statistique de l'Efficacité

In [ ]:
#On importe les différentes librairies dont on aura besoin
import polars as pl
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact
from math import log, exp, sqrt

###  1. FONCTION DE CHARGEMENT HYBRIDE (Polars -> Pandas)
def load_and_merge_robust(file1, file2):
    """
    Charge deux CSV avec Polars (plus tolérant sur les types),
    harmonise les colonnes conflictuelles en texte, concatène,
    puis renvoie un DataFrame Pandas propre.
    """
    try:
        # Lecture large pour bien détecter les types
        df1 = pl.read_csv(file1, infer_schema_length=10000)
        df2 = pl.read_csv(file2, infer_schema_length=10000)

        # Harmonisation : Si une colonne a un type différent entre les 2 fichiers (ex: Int vs Str),
        # on convertit tout en String (Utf8) pour éviter le crash.
        common_cols = set(df1.columns) & set(df2.columns)
        for col in common_cols:
            if df1[col].dtype != df2[col].dtype:
                df1 = df1.with_columns(pl.col(col).cast(pl.Utf8))
                df2 = df2.with_columns(pl.col(col).cast(pl.Utf8))

        # Concaténation avec 'diagonal' pour gérer l'ordre différent des colonnes (ex: dans DS)
        df_concat = pl.concat([df1, df2], how="diagonal")

        # Retourne un DataFrame PANDAS standard pour la suite de votre analyse
        return df_concat.to_pandas()

    except Exception as e:
        print(f"Erreur critique lors du chargement de {file1} et {file2} : {e}")
        return pd.DataFrame()
        #Cela permet de corriger des erreurs types rencontré auparavant

pd.set_option('future.no_silent_downcasting', True) # permet de corriger le message d'erreur du à une nouvelle version de pandas à la cellule 8

In [ ]:
###  2. CHARGEMENT DES DONNÉES
print("Chargement et fusion des fichiers...")

# Assurez-vous que vos fichiers s'appellent bien ainsi ou ajustez les chemins
dm = load_and_merge_robust("dm1.csv", "dm2.csv")
lb = load_and_merge_robust("lb1.csv", "lb2.csv")
ex = load_and_merge_robust("ex1.csv", "ex2.csv")
ds = load_and_merge_robust("ds1.csv", "ds2.csv")

print(f"Données chargées : DM={dm.shape}, LB={lb.shape}, EX={ex.shape}, DS={ds.shape}")

Chargement et fusion des fichiers...
Données chargées : DM=(411, 20), LB=(17977, 27), EX=(4435, 19), DS=(1492, 15)


In [ ]:
###  3. NETTOYAGE PRÉALABLE
# Nettoyage des USUBJID sur toutes les tables
for df_name in ["dm", "lb", "ex", "ds"]:
    # On récupère l'objet DataFrame via son nom
    df = globals()[df_name]
    if "USUBJID" in df.columns:
        # Conversion string + strip (nettoyage espaces)
        df["USUBJID"] = df["USUBJID"].astype(str).str.strip()
        # Mise à jour de la variable globale
        globals()[df_name] = df

In [ ]:
###  4. POPULATION ITT
# Sélection des sujets et du bras de traitement
dm_subj = dm[["USUBJID", "ARMCD"]].drop_duplicates()

# Définition du contexte (IN = Hospitalisé, OUT = Ambulatoire) selon le préfixe ID
dm_subj["SETTING"] = np.select(
    [
        dm_subj["USUBJID"].str.startswith("01_"),
        dm_subj["USUBJID"].str.startswith("02_"),
    ],
    ["IN", "OUT"],
    default="UNK"
)

# On ne garde que les bras d'intérêt
dm_subj = dm_subj[dm_subj["ARMCD"].isin(["BUPNAL", "CLON"])]

In [ ]:
###  5. PRÉPARATION LABOS (LB)
lb_u = lb.copy()

# Conversion numérique forcée (les erreurs deviennent NaN)
lb_u["LBDY_NUM"] = pd.to_numeric(lb_u["LBDY"], errors="coerce")
lb_u["VISITNUM_NUM"] = pd.to_numeric(lb_u["VISITNUM"], errors="coerce")

# Création d'une colonne RÉSULTAT unique et propre
lb_u["RES"] = (
    lb_u["LBSTRESC"]
    .combine_first(lb_u["LBORRES"])
    .astype(str)
    .str.strip()
    .str.upper()
)

# Filtre : Urines, Opiacés/Morphine uniquement
lb_u = lb_u[
    (lb_u["LBSPEC"] == "URINE") &
    (lb_u["LBTESTCD"].isin(["OPIATE", "MORPHINE"])) &
    (lb_u["RES"].notna())
][["USUBJID", "LBTESTCD", "RES", "LBDY_NUM", "VISITNUM_NUM"]]
print(f"Tests urinaires pertinents trouvés : {len(lb_u)}")

Tests urinaires pertinents trouvés : 2189


In [ ]:
###  6. ANALYSE "IN" (HOSPITALISÉ)
# Critère : Test négatif au dernier jour (13 ou 14)
lb_in = lb_u[
    lb_u["USUBJID"].str.startswith("01_") &
    lb_u["LBDY_NUM"].isin([13, 14])
]

# Trouver le dernier jour disponible pour chaque patient (13 ou 14)
last_in = (
    lb_in.groupby("USUBJID")["LBDY_NUM"]
    .max()
    .reset_index(name="LAST_DAY")
)

lb_in_last = (
    lb_in.merge(last_in, on="USUBJID", how="inner")
    .query("LBDY_NUM == LAST_DAY")
)

# Succès si au moins un test est NEGATIVE ce jour-là
succ_in = (
    lb_in_last
    .groupby("USUBJID")["RES"]
    .apply(lambda x: (x == "NEGATIVE").any())
    .reset_index(name="SUCCESS_IN")
)
print("Calcul des critères de succès terminé.")

Calcul des critères de succès terminé.


In [ ]:
###  7. ANALYSE "OUT" (AMBULATOIRE)
# Critère : Test négatif et aucun positif à la dernière visite prévue
lb_out = lb_u[
    lb_u["USUBJID"].str.startswith("02_") &
    (lb_u["LBTESTCD"] == "OPIATE")
]

# Récupérer les numéros de visite correspondant aux jours théoriques 13/14
eot_visitnums = (
    lb_out[
        lb_out["LBDY_NUM"].isin([13, 14]) &
        lb_out["VISITNUM_NUM"].notna()
    ][["VISITNUM_NUM"]]
    .drop_duplicates()
)

lb_out_eot = lb_out.merge(eot_visitnums, on="VISITNUM_NUM", how="inner")

# Trouver la dernière visite réelle
last_out = (
    lb_out_eot.groupby("USUBJID")["VISITNUM_NUM"]
    .max()
    .reset_index(name="LAST_EOT_VISIT")
)

lb_out_last = (
    lb_out_eot.merge(last_out, on="USUBJID", how="inner")
    .query("VISITNUM_NUM == LAST_EOT_VISIT")
)

# Succès : Avoir du négatif ET ne pas avoir du positif
succ_out = (
    lb_out_last.groupby("USUBJID")["RES"]
    .agg(
        any_neg=lambda x: (x == "NEGATIVE").any(),
        any_pos=lambda x: (x == "POSITIVE").any()
    )
    .reset_index()
)

succ_out["SUCCESS_OUT"] = succ_out["any_neg"] & ~succ_out["any_pos"]
succ_out = succ_out[["USUBJID", "SUCCESS_OUT"]]

In [ ]:
###  8. CONSOLIDATION ET RÉSULTATS
analysis = (
    dm_subj
    .merge(succ_in, on="USUBJID", how="left")
    .merge(succ_out, on="USUBJID", how="left")
)

# Remplir les NaN par False (Echec si pas de données aux jours critiques)
analysis["SUCCESS_IN"] = analysis["SUCCESS_IN"].fillna(False).astype(bool)
analysis["SUCCESS_OUT"] = analysis["SUCCESS_OUT"].fillna(False).astype(bool)

# Colonne finale SUCCESS selon le setting
analysis["SUCCESS"] = np.where(
    analysis["SETTING"] == "IN",
    analysis["SUCCESS_IN"],
    np.where(
        analysis["SETTING"] == "OUT",
        analysis["SUCCESS_OUT"],
        False
    )
)

summary = (
    analysis
    .groupby(["SETTING", "ARMCD"])
    .agg(
        n_randomized=("SUCCESS", "size"),
        n_success=("SUCCESS", "sum")
    )
    .reset_index()
)
summary["pct"] = ((summary["n_success"] / summary["n_randomized"]) * 100).round(0).astype(int)
summary = summary.sort_values(["SETTING", "ARMCD"])
print("\n=== TABLEAU RÉCAPITULATIF (EFFICACITÉ) ===")
print(summary)


=== TABLEAU RÉCAPITULATIF (EFFICACITÉ) ===
  SETTING   ARMCD  n_randomized  n_success  pct
0      IN  BUPNAL            77         59   77
1      IN    CLON            36          8   22
2     OUT  BUPNAL           156         45   29
3     OUT    CLON            74          4    5


In [ ]:
###  9. STATISTIQUES (CHI2, FISHER, OR)

def contingency_2x2(df: pd.DataFrame, setting: str):
    """Calcule la table de contingence pour un setting donné"""
    d = df[df["SETTING"] == setting][["ARMCD", "SUCCESS"]]

    # On compte les succès/échecs pour chaque bras
    a = len(d[(d["ARMCD"]=="BUPNAL") & (d["SUCCESS"])])     # Bras A Succès
    b = len(d[(d["ARMCD"]=="BUPNAL") & (~d["SUCCESS"])])    # Bras A Echec
    c = len(d[(d["ARMCD"]=="CLON") & (d["SUCCESS"])])       # Bras B Succès
    d_val = len(d[(d["ARMCD"]=="CLON") & (~d["SUCCESS"])])  # Bras B Echec

    return a, b, c, d_val

def or_ci95(a, b, c, d_val):
    """Calcule l'Odds Ratio et son IC à 95%"""
    # Correction de continuité si une case est vide
    if min(a, b, c, d_val) == 0:
        a, b, c, d_val = a+0.5, b+0.5, c+0.5, d_val+0.5

    OR = (a * d_val) / (b * c)
    se = sqrt(1/a + 1/b + 1/c + 1/d_val) # Erreur standard
    return OR, exp(log(OR) - 1.96*se), exp(log(OR) + 1.96*se)

def run_stats(df: pd.DataFrame, setting: str):
    a, b, c, d_val = contingency_2x2(df, setting)

    # Test du Chi2
    chi2, p_chi, dof, _ = chi2_contingency([[a, b], [c, d_val]], correction=False)

    # Odds Ratio
    OR, lo, hi = or_ci95(a, b, c, d_val)

    # Test Exact de Fisher
    _, p_fisher = fisher_exact([[a, b], [c, d_val]])

    return {
        "SETTING": setting,
        "n_BUP_succ": a, "n_BUP_fail": b,
        "n_CLON_succ": c, "n_CLON_fail": d_val,
        "OR": round(OR, 2),
        "OR_CI_Low": round(lo, 2),
        "OR_CI_High": round(hi, 2),
        "p_Chi2": round(p_chi, 4),
        "p_Fisher": round(p_fisher, 4)
    }

# On crée le DataFrame des résultats
stats_results = pd.DataFrame([
    run_stats(analysis, "IN"),
    run_stats(analysis, "OUT")
])

print("\n RÉSULTATS STATISTIQUES ")
# On affiche les résultats transposés (.T) pour qu'ils soient lisibles
print(stats_results.set_index("SETTING").T)


=== RÉSULTATS STATISTIQUES ===
SETTING         IN     OUT
n_BUP_succ   59.00   45.00
n_BUP_fail   18.00  111.00
n_CLON_succ   8.00    4.00
n_CLON_fail  28.00   70.00
OR           11.47    7.09
OR_CI_Low     4.45    2.44
OR_CI_High   29.56   20.59
p_Chi2        0.00    0.00
p_Fisher      0.00    0.00
